In [4]:
import json
import os
import unittest
import uuid


class Product:
    def __init__(self, product_id, name, price, stock):
        if not product_id or type(product_id) != str:
            raise ValueError("Product ID valid string honi chahiye")
        if not name or type(name) != str:
            raise ValueError("Product name valid string hona chahiye")
        if type(price) not in (int, float) or price < 0:
            raise ValueError("Price negative nahi ho sakti")
        if type(stock) != int or stock < 0:
            raise ValueError("Stock non-negative integer hona chahiye")

        self.product_id = product_id
        self.name = name
        self.price = float(price)
        self.stock = stock

    def to_dict(self):
        return {
            "product_id": self.product_id,
            "name": self.name,
            "price": self.price,
            "stock": self.stock
        }


class Inventory:
    def __init__(self):
        self._products = {}

    def add_product(self, product):
        if not isinstance(product, Product):
            raise TypeError("Sirf Product object add kiya ja sakta hai")
        if product.product_id in self._products:
            raise ValueError(f"Duplicate product ID: '{product.product_id}'")
        self._products[product.product_id] = product

    def get_product(self, product_id):
        if product_id not in self._products:
            raise KeyError(f"Product '{product_id}' inventory mein maujood nahi hai")
        return self._products[product_id]

    @classmethod
    def from_json(cls, filepath):
        if not os.path.exists(filepath):
            raise FileNotFoundError(f"Catalog file nahi mili: {filepath}")

        with open(filepath, "r", encoding="utf-8") as file:
            raw_data = json.load(file)

        if type(raw_data) != list:
            raise ValueError("Catalog JSON ek list honi chahiye")

        inv = cls()
        for item in raw_data:
            prod = Product(
                product_id=item["product_id"],
                name=item["name"],
                price=item["price"],
                stock=item["stock"]
            )
            inv.add_product(prod)
        return inv

    def snapshot(self):
        # Rollback ke liye current stock levels save karein
        return {pid: prod.stock for pid, prod in self._products.items()}

    def restore(self, stock_snapshot):
        for pid, old_stock in stock_snapshot.items():
            if pid in self._products:
                self._products[pid].stock = old_stock


class Cart:
    def __init__(self):
        self._items = {}  # {product_id: quantity}
        self._discount_percent = 0.0

    def add_item(self, product_id, quantity):
        if not product_id or type(product_id) != str:
            raise ValueError("Product ID valid string honi chahiye")
        if type(quantity) != int or quantity <= 0:
            raise ValueError("Quantity positive integer honi chahiye")

        if product_id not in self._items:
            self._items[product_id] = 0
        self._items[product_id] += quantity

    def apply_discount(self, discount_percent):
        if type(discount_percent) not in (int, float) or not (0 <= discount_percent <= 100):
            raise ValueError("Discount percentage 0 aur 100 ke darmiyan honi chahiye")
        self._discount_percent = float(discount_percent)

    @property
    def items(self):
        return dict(self._items)

    @property
    def discount_percent(self):
        return self._discount_percent


class Order:
    def __init__(self, order_id, line_items, subtotal, discount_percent, total):
        self.order_id = order_id
        self.line_items = line_items
        self.subtotal = round(subtotal, 2)
        self.discount_percent = round(discount_percent, 2)
        self.total = round(total, 2)

    def to_dict(self):
        return {
            "order_id": self.order_id,
            "items": self.line_items,
            "subtotal": self.subtotal,
            "discount_percent": self.discount_percent,
            "total": self.total
        }


class OrderProcessor:
    def __init__(self, inventory, orders_filepath="orders.json"):
        if not isinstance(inventory, Inventory):
            raise TypeError("Inventory object pass karna zaroori hai")
        self.inventory = inventory
        self.orders_filepath = orders_filepath

    def _persist_order(self, order):
        existing_orders = []
        if os.path.exists(self.orders_filepath):
            with open(self.orders_filepath, "r", encoding="utf-8") as file:
                content = file.read().strip()
                if content:
                    existing_orders = json.loads(content)
                    if type(existing_orders) != list:
                        raise ValueError("Orders JSON file ka format list hona chahiye")

        existing_orders.append(order.to_dict())

        with open(self.orders_filepath, "w", encoding="utf-8") as file:
            json.dump(existing_orders, file, indent=2)

    def place_order(self, cart):
        if not isinstance(cart, Cart):
            raise TypeError("Cart object pass karna zaroori hai")

        cart_items = cart.items
        if len(cart_items) == 0:
            raise ValueError("Khali cart par order place nahi ho sakta")

        # Step 1: Inventory change karne se pehle saare items aur stock validate karein
        validated_lines = []
        subtotal = 0.0

        for pid, qty in cart_items.items():
            if type(qty) != int or qty <= 0:
                raise ValueError(f"Invalid quantity ({qty}) product '{pid}' ke liye")

            product = self.inventory.get_product(pid)
            if qty > product.stock:
                raise ValueError(
                    f"Out of stock: '{pid}' ke {qty} units maange gaye, sirf {product.stock} maujood hain"
                )

            line_total = product.price * qty
            subtotal += line_total
            validated_lines.append({
                "product_id": pid,
                "name": product.name,
                "unit_price": product.price,
                "quantity": qty,
                "line_total": round(line_total, 2)
            })

        discount_amount = subtotal * (cart.discount_percent / 100.0)
        final_total = subtotal - discount_amount
        unique_order_id = "ORD-" + uuid.uuid4().hex[:8].upper()

        order = Order(
            order_id=unique_order_id,
            line_items=validated_lines,
            subtotal=subtotal,
            discount_percent=cart.discount_percent,
            total=final_total
        )

        # Step 2: Atomic stock update + JSON persistence (fail hone par rollback)
        previous_stocks = self.inventory.snapshot()
        try:
            for pid, qty in cart_items.items():
                product = self.inventory.get_product(pid)
                product.stock -= qty

            self._persist_order(order)
        except Exception:
            # Agar file save karne mein koi error aaye to stock wapas purani state par le aayein
            self.inventory.restore(previous_stocks)
            raise

        return order


# ==========================================
# UNIT TESTS (unittest)
# ==========================================

class TestOrderFulfillmentSystem(unittest.TestCase):
    def setUp(self):
        self.catalog_file = "test_catalog.json"
        self.orders_file = "test_orders.json"

        sample_catalog = [
            {"product_id": "P1", "name": "Wireless Mouse", "price": 100, "stock": 5},
            {"product_id": "P2", "name": "USB Hub", "price": 50, "stock": 2}
        ]
        with open(self.catalog_file, "w", encoding="utf-8") as f:
            json.dump(sample_catalog, f, indent=2)

        if os.path.exists(self.orders_file):
            os.remove(self.orders_file)

        self.inventory = Inventory.from_json(self.catalog_file)
        self.processor = OrderProcessor(self.inventory, self.orders_file)

    def tearDown(self):
        for path in (self.catalog_file, self.orders_file):
            if os.path.exists(path):
                os.remove(path)

    def test_successful_order_with_discount_and_stock_update(self):
        cart = Cart()
        cart.add_item("P1", 2)
        cart.apply_discount(10)

        order = self.processor.place_order(cart)

        self.assertEqual(order.total, 180.0)
        self.assertEqual(self.inventory.get_product("P1").stock, 3)
        self.assertTrue(os.path.exists(self.orders_file))

        with open(self.orders_file, "r", encoding="utf-8") as f:
            saved_orders = json.load(f)
        self.assertEqual(len(saved_orders), 1)
        self.assertEqual(saved_orders[0]["order_id"], order.order_id)

    def test_reject_out_of_stock_without_partial_deduction(self):
        cart = Cart()
        cart.add_item("P1", 2)   # Valid
        cart.add_item("P2", 10)  # Out of stock (only 2 available)

        with self.assertRaises(ValueError):
            self.processor.place_order(cart)

        # P1 ka stock bhi kam nahi hona chahiye
        self.assertEqual(self.inventory.get_product("P1").stock, 5)
        self.assertEqual(self.inventory.get_product("P2").stock, 2)

    def test_reject_invalid_quantities_and_discounts(self):
        cart = Cart()
        with self.assertRaises(ValueError):
            cart.add_item("P1", 0)
        with self.assertRaises(ValueError):
            cart.add_item("P1", -3)
        with self.assertRaises(ValueError):
            cart.apply_discount(120)

    def test_rollback_inventory_when_persistence_fails(self):
        # Orders filepath ko ek directory path bana dein taake open() fail ho jaye
        bad_processor = OrderProcessor(self.inventory, ".")
        cart = Cart()
        cart.add_item("P1", 3)

        with self.assertRaises(Exception):
            bad_processor.place_order(cart)

        # Persistence fail hone par P1 ka stock wapas 5 hona chahiye
        self.assertEqual(self.inventory.get_product("P1").stock, 5)


if __name__ == "__main__":
    # Sample catalog file generate karein (as per instructions)
    with open("catalog.json", "w", encoding="utf-8") as f:
        json.dump([
            {"product_id": "P1", "name": "Laptop Stand", "price": 100, "stock": 5},
            {"product_id": "P2", "name": "Mechanical Keyboard", "price": 250, "stock": 10}
        ], f, indent=2)

    unittest.main(argv=["first-arg-is-ignored"], exit=False)

....
----------------------------------------------------------------------
Ran 4 tests in 0.111s

OK
